# Papcorns — Data Scientist Technical Assessment

**Approach:** All core metrics (Tasks 1–6) are computed via SQL queries executed with `pandas.read_sql`. Python is used only for orchestration, post-processing (e.g. median over a GROUP BY), and visualization. The pLTV bonus (Task 8) is a cohort-based statistical estimate with a brief ML extension.

## 0. Setup

In [2]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

conn = sqlite3.connect("papcorns.sqlite")

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

print("Connected. Tables:", q("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist())

Connected. Tables: ['users', 'user_events']


### Quick data exploration

In [4]:
print("users (first 5)")
display(q("SELECT * FROM users LIMIT 5"))

print("\nuser_events (first 10)")
display(q("SELECT * FROM user_events LIMIT 10"))

print("\nRow counts")
display(q("""
    SELECT 'users'       AS tbl, COUNT(*) AS rows FROM users
    UNION ALL
    SELECT 'user_events' AS tbl, COUNT(*) AS rows FROM user_events
"""))

print("\nEvent types")
display(q("SELECT event_name, COUNT(*) AS cnt FROM user_events GROUP BY event_name ORDER BY cnt DESC"))

users (first 5)


,id,created_at,attribution_source,country,name
0,1,2024-05-07T00:00:00,instagram,US,Eve Brown
1,2,2024-10-12T00:00:00,instagram,NL,Frank Moore
2,3,2024-10-15T00:00:00,tiktok,TR,Ivy Anderson
3,4,2024-08-28T00:00:00,tiktok,TR,Alice Brown
4,5,2024-04-03T00:00:00,organic,NL,Bob Moore



user_events (first 10)


,id,created_at,user_id,event_name,amount_usd
0,1,2024-05-07T00:00:00,1,app_install,NaN
1,2,2024-05-12T00:00:00,1,trial_started,NaN
2,3,2024-05-24T00:00:00,1,trial_cancelled,NaN
3,4,2024-10-12T00:00:00,2,app_install,NaN
4,5,2024-10-13T00:00:00,2,trial_started,NaN
5,6,2024-10-20T00:00:00,2,subscription_started,8.99
6,7,2024-11-19T00:00:00,2,subscription_renewed,8.99
7,8,2024-12-19T00:00:00,2,subscription_renewed,8.99
8,9,2025-01-18T00:00:00,2,subscription_renewed,8.99
9,10,2025-02-12T00:00:00,2,subscription_cancelled,NaN



Row counts


,tbl,rows
0,users,1002
1,user_events,3486



Event types


,event_name,cnt
0,app_install,1002
1,subscription_renewed,750
2,trial_started,682
3,subscription_started,481
4,subscription_cancelled,370
5,trial_cancelled,201
